# The genome-wide landscape of dynamic RBP target-site dysregulation

This notebook is a **reproducible, end-to-end tutorial** for computing BRIDGE scores on GWAS variants before and after applying the initial mutation (REF→ALT), then parsing & annotating the results and generating downstream plots.

## What you will get at the end
- Per-variant BRIDGE scores:
  - `ref_score`  (variation_mode=`before`, i.e., reference allele context)
  - `alt_score`  (variation_mode=`after`,  i.e., mutated allele context)
- A tidy table saved as:
  - `./results/reproducibility/gwas_resources/mutation_scores.csv`
  - `./results/reproducibility/gwas_resources/mutation_scores_with_gene.csv`
  - `./results/reproducibility/gwas_resources/match_GWAS.csv`

## Execution modes  
1. **Smoke test (fast, just one dataset)**  
   Runs BRIDGE on a **small subset** of variants to validate environment + I/O contracts.
2. **Full-scale run (slow, the large-scale analysis)**  
   Runs BRIDGE on **all FASTA files** under `dataset_variant/` and produces the full set of score files.

```{important}
Prerequisite: download and unzip `dataset_variant/` from figshare into the BRIDGE repo root. Then run the commands in this notebook to produce the baseline GWAS variant scores. All downstream parsing, annotation and plots depend on these outputs.
```

## Input/Output specification

### Input: `dataset_variant/*.fa`
Each FASTA record corresponds to one variant instance. The header encodes metadata that will be parsed later.

**Expected header format (example):**
```
>variant_1 chr1:27891903-27892003(-)[3_prime_UTR_variant]{NA} 27891953:T>A ENSG00000117748[0.002]{rs6666467}
```

**Field semantics**
```
chr1:27891903-27892003(-)  → genomic window and strand
[3_prime_UTR_variant]      → region / consequence label
{NA}                       → clinical significance (may be empty/NA)
27891953:T>A               → position and REF→ALT alleles
ENSG...                    → Ensembl gene ID
[0.002]                    → minor allele frequency (MAF)
{rs...}                    → rsID
```

### Scoring script: `variant_aware.py`
This notebook calls:
- `variation_mode=before` → score **reference** (REF) context ⇒ `ref_score`
- `variation_mode=after`  → score **mutated** (ALT) context   ⇒ `alt_score`

**CLI parameters (must be provided)**
- `--variation_mode` : `before` or `after`
- `--fasta_sequence_path` : input FASTA file path
- `--Transformer_path` : local directory of the pretrained Transformer/“RBPformer” checkpoint (tokenizer + model)
- `--model_save_path` : directory containing trained BRIDGE weights/checkpoints
- `--variant_out_file` : output text file path (one score per variant)
- `--device` : `cpu` or `cuda:num`

### Output: score text files
Full-scale runs produce:
- `./results/reproducibility/gwas_resources/before_mut/<RBP>_<CELL>_before_mut.txt`
- `./results/reproducibility/gwas_resources/after_mut/<RBP>_<CELL>_after_mut.txt`

Each line is expected to contain `Prediction_score: <float>` and the original header.

### Parsed table schema
`./results/reproducibility/gwas_resources/mutation_scores.csv` contains:
- `rsid`, `rbp`, `cell`, `MAF`
- `alt_score`, `ref_score`, `delta_score = alt_score - ref_score`
- `region`, `clin_sig`, `gene`, `gene_name`
- `chr`, `start_pos`, `end_pos`, `strand`


### Appendix — Score line examples & parsing caveats
The following is the original detailed note about how `alt_score`/`ref_score` are extracted from score files.

In [1]:
"""
Notebook utilities for GWAS scoring.

Design goals:
- Keep downstream plotting cells intact (stable filenames + schemas).
- Make I/O contracts explicit (FASTA header format, score line format).
- Provide both a smoke-test path and a full-scale run path.
"""

from __future__ import annotations

import os
import re
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd


# ---------------------------------------------------------------------
# 1) Parsing contract for BRIDGE score files
# ---------------------------------------------------------------------
# The scoring output is expected to preserve the original FASTA header and
# append "Prediction_score: <float>" at the end of each line.
#
# IMPORTANT:
#   We assume 1-to-1 line alignment between *_after_mut.txt and *_before_mut.txt,
#   meaning the k-th line in both files refers to the same variant instance.
SCORE_HEADER_PATTERN = re.compile(
    r"variant_\d+ chr(?P<chr>[^:]+):(?P<start>\d+)-(?P<end>\d+)\((?P<strand>[+-])\)"
    r"\[(?P<region>[^\]]+)\]\{(?P<clin_sig>[^\}]*)\} (?P<pos>\d+):(?P<ref>[ACGT])>(?P<alt>[ACGT]) "
    r"(?P<gene>ENSG\d+)\[(?P<maf>[-+eE0-9.]+)\]\{(?P<rsid>[^\}]+)\}"
)

def _extract_prediction_score(line: str) -> float:
    """Extract the trailing float after the token 'Prediction_score:'."""
    # Example suffix: "Prediction_score: 0.3616018295288086"
    try:
        return float(line.strip().split("Prediction_score:")[-1])
    except Exception as e:
        raise ValueError(f"Failed to parse Prediction_score from line: {line}") from e


def parse_bridge_before_after(
    after_mut_dir: str,
    before_mut_dir: str,
    *,
    strict: bool = True,
) -> pd.DataFrame:
    """
    Parse BRIDGE outputs into a tidy DataFrame.

    Inputs
    ------
    after_mut_dir:
        Directory containing files named: <RBP>_<CELL>_after_mut.txt
    before_mut_dir:
        Directory containing matching:    <RBP>_<CELL>_before_mut.txt
    strict:
        If True, raise on mismatches (missing files, unparsable lines).
        If False, skip problematic lines/files and continue.

    Returns
    -------
    pd.DataFrame with columns:
        rsid, rbp, cell, MAF, alt_score, ref_score, region, clin_sig, gene,
        chr, start_pos, end_pos, strand
    """
    results: List[Dict[str, object]] = []

    if not os.path.isdir(after_mut_dir):
        raise FileNotFoundError(f"after_mut_dir not found: {after_mut_dir}")
    if not os.path.isdir(before_mut_dir):
        raise FileNotFoundError(f"before_mut_dir not found: {before_mut_dir}")

    for filename in sorted(os.listdir(after_mut_dir)):
        if not filename.endswith("_after_mut.txt"):
            continue

        stem = filename.replace("_after_mut.txt", "")
        if "_" not in stem:
            # Skip any unexpected filenames.
            continue

        rbp, cell = stem.split("_", 1)
        after_path = os.path.join(after_mut_dir, filename)
        before_path = os.path.join(before_mut_dir, f"{stem}_before_mut.txt")

        if not os.path.exists(before_path):
            msg = f"Missing before_mut file: {before_path}"
            if strict:
                raise FileNotFoundError(msg)
            print(msg)
            continue

        with open(after_path, "r", encoding="utf-8", errors="replace") as f_after, \
             open(before_path, "r", encoding="utf-8", errors="replace") as f_before:

            for line_after, line_before in zip(f_after, f_before):
                m = SCORE_HEADER_PATTERN.search(line_after)
                if not m:
                    msg = f"Failed to match header pattern. after_line={line_after.strip()[:200]}"
                    if strict:
                        raise ValueError(msg)
                    print(msg)
                    continue

                try:
                    alt_score = _extract_prediction_score(line_after)
                    ref_score = _extract_prediction_score(line_before)
                    maf_value = float(m.group("maf"))
                except Exception as e:
                    msg = f"Failed to parse scores/MAF. after_line={line_after.strip()[:200]}"
                    if strict:
                        raise
                    print(msg, e)
                    continue

                results.append(
                    {
                        "rsid": m.group("rsid"),
                        "rbp": rbp,
                        "cell": cell,
                        "MAF": maf_value,
                        "alt_score": alt_score,
                        "ref_score": ref_score,
                        "region": m.group("region"),
                        "clin_sig": m.group("clin_sig"),
                        "gene": m.group("gene"),
                        "chr": f"chr{m.group('chr')}",
                        "start_pos": int(m.group("start")),
                        "end_pos": int(m.group("end")),
                        "strand": m.group("strand"),
                    }
                )

    df = pd.DataFrame(results)
    if len(df) == 0:
        print("WARNING: parsed DataFrame is empty. Check input directories and filename patterns.")
    return df


def summarize_scores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a lightweight summary of BRIDGE scores.

    Adds `delta_score = alt_score - ref_score` and returns a small table with:
    - counts
    - basic statistics
    This is designed to work for both smoke tests and full runs.
    """
    if df.empty:
        return pd.DataFrame()

    df = df.copy()
    df["delta_score"] = df["alt_score"] - df["ref_score"]

    summary = pd.DataFrame(
        {
            "n_rows": [len(df)],
            "n_unique_rsid": [df["rsid"].nunique()],
            "alt_score_mean": [df["alt_score"].mean()],
            "ref_score_mean": [df["ref_score"].mean()],
            "delta_mean": [df["delta_score"].mean()],
            "delta_median": [df["delta_score"].median()],
        }
    )
    return summary


## Step 1 — Run BRIDGE on smoke test or full-scale dataset

You may choose either option **A** or **B** below, depending on whether you want to perform a quick functionality check or a full-scale production run.

### (A) Compute BRIDGE scores on a small mutated (ALT) dataset and reference (REF) dataset

`after_mut` stores BRIDGE scores on the **mutated** context (used as `alt_score`).
`before_mut` stores BRIDGE scores on the **reference** context (used as `ref_score`).

> Tip: start with the smoke-test section below if you want a fast validation run.

In [ ]:
%%bash
set -euo pipefail

# Single-file example (ALT context).

BRIDGE_HOME=../../
cd $BRIDGE_HOME

python variant_aware.py   \
    --variation_mode after   \
    --fasta_sequence_path ./dataset_variant/AUH_HepG2.fa   \
    --Transformer_path ./RBPformer   \
    --model_save_path ./results/model   \
    --variant_out_file ./results/reproducibility/gwas_resources/after_mut/AUH_HepG2_after_mut.txt   \
    --device cuda:3


INFO: Running in after_variation mode
INFO: Loading FASTA from dataset_variant/AUH_HepG2.fa
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this mo

In [ ]:
%%bash
set -euo pipefail

# Single-file example (REF context).

BRIDGE_HOME=../../
cd $BRIDGE_HOME

python variant_aware.py  \
    --variation_mode before   \
    --fasta_sequence_path ./dataset_variant/AUH_HepG2.fa   \
    --Transformer_path ./RBPformer   \
    --model_save_path ./results/model   \
    --variant_out_file ./results/reproducibility/gwas_resources/before_mut/AUH_HepG2_before_mut.txt   \
    --device cuda:3

INFO: Running in before_variation mode
INFO: Loading FASTA from dataset_variant/AUH_HepG2.fa
/home/wangyubo/softwares/anaconda3/envs/BRIDGE/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at RBPformer and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this m

### (B) Full-scale run (score all datasets)

This section runs BRIDGE scoring for **all** datasets under `dataset_variant/*.fa`.

Outputs:
- `./results/reproducibility/gwas_resources/after_mut/<BASENAME>_after_mut.txt`
- `./results/reproducibility/gwas_resources/before_mut/<BASENAME>_before_mut.txt`

> Tip: You can set `DEVICE=cuda:3` (or any GPU index) in the cell below.


In [ ]:
%%bash
set -euo pipefail

# Full-scale batch scoring over all FASTA files in dataset_variant/.
# This is the canonical "production" path for generating BRIDGE scores
# before/after the initial mutation.

BRIDGE_HOME=../../
cd $BRIDGE_HOME

DEVICE=${DEVICE:-cuda:3}

mkdir -p ./results/reproducibility/gwas_resources/after_mut ./results/reproducibility/gwas_resources/before_mut

for fasta in ./dataset_variant/*.fa; do
  base=$(basename "$fasta" .fa)
  echo "[INFO] Scoring: ${base}  (device=${DEVICE})"

  python variant_aware.py     --variation_mode after     --fasta_sequence_path "$fasta"     --Transformer_path ./RBPformer     --model_save_path ./results/model     --variant_out_file "./results/reproducibility/gwas_resources/after_mut/${base}_after_mut.txt"     --device ${DEVICE}
  python variant_aware.py     --variation_mode before     --fasta_sequence_path "$fasta"     --Transformer_path ./RBPformer     --model_save_path ./results/model     --variant_out_file "./results/reproducibility/gwas_resources/before_mut/${base}_before_mut.txt"     --device ${DEVICE}
done

echo "[DONE] Full-scale scoring complete."


## Step 2 — Parse BRIDGE score files into a tidy table

We read paired score files from:
- `./results/reproducibility/gwas_resources/after_mut/`  (ALT / mutated context → `alt_score`)
- `./results/reproducibility/gwas_resources/before_mut/` (REF / reference context → `ref_score`)

and parse the metadata embedded in the header (rsid, region, gene, MAF, etc.) into a single CSV.

**Output**
- `./results/reproducibility/gwas_resources/mutation_scores.csv`

**Important contract**
- Files must be named `<RBP>_<CELL>_after_mut.txt` and `<RBP>_<CELL>_before_mut.txt`.
- Lines are assumed to be aligned 1-to-1 across the paired files.


### Step 2A — Parse full-scale outputs into a single table
These paths are the stable contract used by downstream plotting cells.

In [5]:
try:
    from IPython.display import display
except Exception:
    display = print  # fallback

after_mut_dir = "../../results/reproducibility/gwas_resources/after_mut"
before_mut_dir = "../../results/reproducibility/gwas_resources/before_mut"
output_mutation_scores = "../../results/reproducibility/gwas_resources/mutation_scores.csv"

df = parse_bridge_before_after(after_mut_dir, before_mut_dir, strict=True)

# Convenience column used throughout analysis.
df["delta_score"] = df["alt_score"] - df["ref_score"]

# Persist for downstream steps.
df.to_csv(output_mutation_scores, index=False)

display(df.head(10))
display(summarize_scores(df))

print(f"Saved parsed results to: {output_mutation_scores}")


,rsid,rbp,cell,MAF,alt_score,ref_score,region,clin_sig,gene,chr,start_pos,end_pos,strand,delta_score
0,rs116002608,AUH,HepG2,0.001,-1.644250,0.165890,synonymous_variant,benign,ENSG00000187608,chr1,1014401,1014501,+,-1.810140
1,rs2232460,AUH,HepG2,0.339,-3.908626,-0.168245,synonymous_variant,NA,ENSG00000162413,chr1,6599395,6599495,-,-3.740381
2,rs2232460,AUH,HepG2,0.339,-4.906653,-9.134668,non_coding_transcript_exon_variant,NA,ENSG00000295286,chr1,6599395,6599495,+,4.228015
3,rs149512075,AUH,HepG2,0.005,-14.313357,-14.266900,5_prime_UTR_variant,NA,ENSG00000142599,chr1,8817541,8817641,-,-0.046457
4,rs41280772,AUH,HepG2,0.017,-10.647985,-8.807708,3_prime_UTR_variant,NA,ENSG00000171603,chr1,9729003,9729103,-,-1.840277
5,rs41280772,AUH,HepG2,0.017,-23.168470,-18.567013,3_prime_UTR_variant,NA,ENSG00000171608,chr1,9729003,9729103,+,-4.601457
6,rs189369210,AUH,HepG2,0.010,-3.153581,-4.073249,5_prime_UTR_variant,NA,ENSG00000142657,chr1,10399029,10399129,+,0.919668
7,chr1:10520396:A:C,AUH,HepG2,0.000,-10.758746,-11.672715,3_prime_UTR_variant,NA,ENSG00000160049,chr1,10460289,10460389,-,0.913969
8,chr1:26497319:T:C,AUH,HepG2,0.000,5.158173,5.170312,3_prime_UTR_variant,NA,ENSG00000142684,chr1,26170778,26170878,+,-0.012139
9,chr1:26497319:T:C,AUH,HepG2,0.000,1.580930,0.472302,intron_variant,NA,ENSG00000236782,chr1,26170778,26170878,-,1.108628


,n_rows,n_unique_rsid,alt_score_mean,ref_score_mean,delta_mean,delta_median
0,1554,647,-7.295303,-5.920635,-1.374668,-0.920518


Saved parsed results to: ../../results/reproducibility/gwas_resources/mutation_scores.csv


### Step 2B — Quick sanity checks for "before vs after" BRIDGE scores

This is the requested "before/after initial mutation" scoring report.
It is intentionally lightweight (no plotting) so it can run everywhere.

In [6]:
try:
    from IPython.display import display
except Exception:
    display = print

if df.empty:
    raise RuntimeError("Parsed score table is empty; cannot compute sanity checks.")

# Basic descriptive statistics
display(df[["alt_score", "ref_score", "delta_score", "MAF"]].describe().T)

# How many variants flip sign after mutation?
sign_flip_rate = ((df["alt_score"] >= 0) != (df["ref_score"] >= 0)).mean()
print(f"Sign-flip rate (alt vs ref) = {sign_flip_rate:.3%}")

# Top |delta| variants (useful for spot-checking)
top_k = 10
display(
    df.assign(abs_delta=df["delta_score"].abs())
      .sort_values("abs_delta", ascending=False)
      .head(top_k)[["rsid", "rbp", "cell", "region", "gene", "MAF", "ref_score", "alt_score", "delta_score"]]
)


,count,mean,std,min,25%,50%,75%,max
alt_score,1554.0,-7.295303,7.844890,-51.862415,-11.858242,-6.441747,-2.069685,14.440907
ref_score,1554.0,-5.920635,7.152513,-50.635330,-9.291387,-4.400947,-1.027438,14.221816
delta_score,1554.0,-1.374668,3.487810,-18.703649,-3.115223,-0.920518,0.727036,10.258034
MAF,1554.0,0.034563,0.707020,-7.843000,0.003500,0.030000,0.142000,0.499300


Sign-flip rate (alt vs ref) = 12.162%


,rsid,rbp,cell,region,gene,MAF,ref_score,alt_score,delta_score
1034,rs1057410,AUH,HepG2,3_prime_UTR_variant,ENSG00000148660,0.1320,-14.820628,-33.524277,-18.703649
793,rs56163360,AUH,HepG2,"intron_variant,non_coding_transcript_variant",ENSG00000310419,0.0050,-8.371350,-24.773363,-16.402013
1346,rs10341,AUH,HepG2,"intron_variant,non_coding_transcript_variant",ENSG00000163597,0.0111,-12.507936,-27.662371,-15.154435
1102,rs36094095,AUH,HepG2,"intron_variant,non_coding_transcript_variant",ENSG00000256007,0.0180,-4.908851,-19.279400,-14.370549
1386,rs923366,AUH,HepG2,"intron_variant,non_coding_transcript_variant",ENSG00000267607,0.4352,-8.024796,-22.274164,-14.249368
1540,rs1894646,AUH,HepG2,3_prime_UTR_variant,ENSG00000189060,0.1365,-8.475801,-22.379993,-13.904192
1067,rs72910089,AUH,HepG2,intron_variant,ENSG00000149091,0.0080,-9.884849,-23.536751,-13.651902
1152,rs1134368,AUH,HepG2,3_prime_UTR_variant,ENSG00000184992,0.3585,-10.041735,-23.661205,-13.619470
848,rs115116946,AUH,HepG2,intron_variant,ENSG00000115677,0.0020,-6.427467,-19.297035,-12.869568
834,rs2217675,AUH,HepG2,5_prime_UTR_variant,ENSG00000154518,0.0380,0.348965,-12.227541,-12.576506


In [7]:
# Prioritise using conda's libraries
!export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:${LD_LIBRARY_PATH}" 

## Step 3 — Annotate Ensembl gene IDs with gene symbols (gene_name)
This step adds a human-readable gene symbol for each Ensembl ID.
It uses the `mygene` service (internet required). If the service is unavailable, we fall back to leaving `gene_name` empty.

In [8]:
import json
import os
import pandas as pd

input_file = output_mutation_scores
output_mutation_scores_with_gene = "../../results/reproducibility/gwas_resources/mutation_scores_with_gene.csv"

df = pd.read_csv(input_file)

# Optional: on-disk cache to avoid repeated lookups across notebook runs.
cache_path = "../../results/reproducibility/gwas_resources/_ensembl_to_symbol_cache.json"
ensembl_to_gene: dict = {}

if os.path.exists(cache_path):
    with open(cache_path, "r", encoding="utf-8") as f:
        ensembl_to_gene = json.load(f)

missing = [g for g in df["gene"].dropna().unique().tolist() if g not in ensembl_to_gene]

try:
    import mygene  # noqa: F401

    if missing:
        mg = mygene.MyGeneInfo()
        query_result = mg.querymany(
            missing,
            scopes="ensembl.gene",
            fields="symbol",
            species="human",
        )
        for item in query_result:
            q = item.get("query")
            if q:
                ensembl_to_gene[q] = item.get("symbol", "")

        # Persist cache for future runs.
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(ensembl_to_gene, f, ensure_ascii=False, indent=2)

except Exception as e:
    print("[WARN] mygene lookup failed; continuing with empty gene_name.")
    print("       Error:", e)

df["gene_name"] = df["gene"].map(lambda g: ensembl_to_gene.get(g, ""))

df.to_csv(output_mutation_scores_with_gene, index=False)
display(df.head(10))
print(f"Saved to: {output_mutation_scores_with_gene}")


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
4 input query terms found no hit:	['ENSG00000302163', 'ENSG00000296575', 'ENSG00000148362', 'ENSG00000299971']


,rsid,rbp,cell,MAF,alt_score,ref_score,region,clin_sig,gene,chr,start_pos,end_pos,strand,delta_score,gene_name
0,rs116002608,AUH,HepG2,0.001,-1.644250,0.165890,synonymous_variant,benign,ENSG00000187608,chr1,1014401,1014501,+,-1.810140,ISG15
1,rs2232460,AUH,HepG2,0.339,-3.908626,-0.168245,synonymous_variant,NaN,ENSG00000162413,chr1,6599395,6599495,-,-3.740381,KLHL21
2,rs2232460,AUH,HepG2,0.339,-4.906653,-9.134668,non_coding_transcript_exon_variant,NaN,ENSG00000295286,chr1,6599395,6599495,+,4.228015,
3,rs149512075,AUH,HepG2,0.005,-14.313357,-14.266900,5_prime_UTR_variant,NaN,ENSG00000142599,chr1,8817541,8817641,-,-0.046457,RERE
4,rs41280772,AUH,HepG2,0.017,-10.647985,-8.807708,3_prime_UTR_variant,NaN,ENSG00000171603,chr1,9729003,9729103,-,-1.840277,CLSTN1
5,rs41280772,AUH,HepG2,0.017,-23.168470,-18.567013,3_prime_UTR_variant,NaN,ENSG00000171608,chr1,9729003,9729103,+,-4.601457,PIK3CD
6,rs189369210,AUH,HepG2,0.010,-3.153581,-4.073249,5_prime_UTR_variant,NaN,ENSG00000142657,chr1,10399029,10399129,+,0.919668,PGD
7,chr1:10520396:A:C,AUH,HepG2,0.000,-10.758746,-11.672715,3_prime_UTR_variant,NaN,ENSG00000160049,chr1,10460289,10460389,-,0.913969,DFFA
8,chr1:26497319:T:C,AUH,HepG2,0.000,5.158173,5.170312,3_prime_UTR_variant,NaN,ENSG00000142684,chr1,26170778,26170878,+,-0.012139,ZNF593
9,chr1:26497319:T:C,AUH,HepG2,0.000,1.580930,0.472302,intron_variant,NaN,ENSG00000236782,chr1,26170778,26170878,-,1.108628,ZNF593OS


Saved to: ../../results/reproducibility/gwas_resources/mutation_scores_with_gene.csv


## Step 4 — Join GWAS p-values by rsid
We add `p_value` from an external GWAS summary statistics file.
The GWAS file must contain columns:
- variant_id  (rsid)
- p_value

In [ ]:
import pandas as pd

mutation_scores_path = output_mutation_scores_with_gene

# Replace with your own GWAS file path, you can download it from figshare
gwas_data_path = "../../results/reproducibility/gwas_resources/34873335-GCST90027164-MONDO_0004976-Build37.f.tsv"

mutation_df = pd.read_csv(mutation_scores_path)
gwas_df = pd.read_csv(gwas_data_path, sep="\t")

required_cols = {"variant_id", "p_value"}
missing_cols = required_cols - set(gwas_df.columns)
if missing_cols:
    raise ValueError(f"GWAS file is missing required columns: {missing_cols}")

rsid_to_pvalue = dict(zip(gwas_df["variant_id"], gwas_df["p_value"]))
mutation_df["p_value"] = mutation_df["rsid"].map(rsid_to_pvalue)

output_path_pvalue = "../../results/reproducibility/gwas_resources/match_GWAS.csv"
mutation_df.to_csv(output_path_pvalue, index=False)

display(mutation_df.head(10))
print(f"Saved GWAS-joined table to: {output_path_pvalue}")


,rsid,rbp,cell,MAF,alt_score,ref_score,region,clin_sig,gene,chr,start_pos,end_pos,strand,delta_score,gene_name,p_value
0,rs116002608,AUH,HepG2,0.001,-1.644250,0.165890,synonymous_variant,benign,ENSG00000187608,chr1,1014401,1014501,+,-1.810140,ISG15,0.9657
1,rs2232460,AUH,HepG2,0.339,-3.908626,-0.168245,synonymous_variant,NaN,ENSG00000162413,chr1,6599395,6599495,-,-3.740381,KLHL21,0.5344
2,rs2232460,AUH,HepG2,0.339,-4.906653,-9.134668,non_coding_transcript_exon_variant,NaN,ENSG00000295286,chr1,6599395,6599495,+,4.228015,NaN,0.5344
3,rs149512075,AUH,HepG2,0.005,-14.313357,-14.266900,5_prime_UTR_variant,NaN,ENSG00000142599,chr1,8817541,8817641,-,-0.046457,RERE,0.9218
4,rs41280772,AUH,HepG2,0.017,-10.647985,-8.807708,3_prime_UTR_variant,NaN,ENSG00000171603,chr1,9729003,9729103,-,-1.840277,CLSTN1,0.7078
5,rs41280772,AUH,HepG2,0.017,-23.168470,-18.567013,3_prime_UTR_variant,NaN,ENSG00000171608,chr1,9729003,9729103,+,-4.601457,PIK3CD,0.7078
6,rs189369210,AUH,HepG2,0.010,-3.153581,-4.073249,5_prime_UTR_variant,NaN,ENSG00000142657,chr1,10399029,10399129,+,0.919668,PGD,0.1842
7,chr1:10520396:A:C,AUH,HepG2,0.000,-10.758746,-11.672715,3_prime_UTR_variant,NaN,ENSG00000160049,chr1,10460289,10460389,-,0.913969,DFFA,0.2681
8,chr1:26497319:T:C,AUH,HepG2,0.000,5.158173,5.170312,3_prime_UTR_variant,NaN,ENSG00000142684,chr1,26170778,26170878,+,-0.012139,ZNF593,0.7051
9,chr1:26497319:T:C,AUH,HepG2,0.000,1.580930,0.472302,intron_variant,NaN,ENSG00000236782,chr1,26170778,26170878,-,1.108628,ZNF593OS,0.7051


Saved GWAS-joined table to: ../../results/reproducibility/gwas_resources/match_GWAS.csv
